# Trabalho prático: simulação e agência inteligente

**Curso:** Pós-Graduação em Inteligência Artificial  
**Módulo:** Agentes Inteligentes  
**Período:** 2026  
**Professor(a):** Vitor Augusto Correa Cortez Almeida  

## 1. Identificação

**Integrantes:**
- Luis Eduardo Alencar Melo
- João Pedro Coelho Barbosa

### Contribuições individuais



- **Luis Eduardo Alencar Melo:** 
1. Planejamento do problema
2. Especificação PEAS
3. Análise do ambiente (Russell & Norvig)
4. Modelagem do ambiente (estrutura do grid e estados)
5. Integração com LLM (Ollama/OpenAI)
6. Implementação do agente coordenador (LLM)
7. Comunicação entre agentes
8. Escrita principal do relatório técnico
- **João Pedro Coelho Barbosa:** 
1. Implementação do agente baseado em objetivos
2. Desenvolvimento do algoritmo de planejamento de caminho
3. Implementação das ações (movimento, coleta, entrega)
4. Execução da simulação
5. Testes com diferentes cenários (SIM_SEED)
6. Validação do comportamento do sistema
7. Revisão da modelagem PEAS
8. Ajustes e debugging da simulação


## 2. Especificação da tarefa (PEAS)

Ambiente: **centro de distribuição em grade discreta** (simulador implementado abaixo).

| Componente | Descrição |
|------------|------------|
| **Performance (medida de desempenho)** | Minimizar o número de passos até coletar todos os pacotes e entregá-los na zona de entrega; evitar movimentos inválidos (colisão com obstáculos). |
| **Environment (ambiente)** | Grade 6×6 com obstáculos fixos, **três pacotes** em células sorteadas (reproduzível com `SIM_SEED`), zona de entrega fixa e robô que se move célula a célula. |
| **Actuators (atuadores)** | Movimento do robô nas quatro direções (uma célula por vez); ações implícitas de **carregar** pacote ao pisar na célula do pacote (quando orientado pelo plano de alto nível) e **entregar** ao chegar à entrega com carga. |
| **Sensors (sensores)** | Estado completo exposto ao simulador: posição do robô, células ocupadas por obstáculos, posições remanescentes de pacotes, se há carga, posição da entrega. O coordenador LLM recebe esse estado em **texto**; o transportador usa a representação estruturada para planejar caminhos. |


## 3. Análise do ambiente (seis dimensões — Russell & Norvig)

| Dimensão | Classificação | Justificativa |
|----------|----------------|---------------|
| Observabilidade | **Totalmente observável** | O estado relevante (posição do robô, pacotes restantes, obstáculos, carga) está disponível integralmente para os agentes a cada decisão de alto nível e para o planejamento de caminho. |
| Agentes | **Multiagente** | Dois agentes autônomos: coordenador (LLM) que define subobjetivos e transportador baseado em objetivos que executa o plano de movimento. |
| Determinismo | **Determinístico** | Dado o estado atual e a ação, o próximo estado é único. O sorteio só define as **posições iniciais** dos pacotes (semente `SIM_SEED`); durante a execução não há aleatoriedade nas transições. |
| Episódico vs sequencial | **Sequencial** | Decisões posteriores dependem de estado acumulado (carga, pacotes já coletados, posição atual); a missão não se decompõe em episódios independentes sem memória. |
| Estático vs dinâmico | **Estático** | Obstáculos e topologia não mudam durante a execução; apenas o estado do robô e dos pacotes evolui em resposta às ações dos agentes. |
| Discreto vs contínuo | **Discreto** | Posições e tempo (passos discretos) são finitos e enumeráveis. |


## 4. Implementação

### Dependências

Na pasta do projeto, com a venv ativada:

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install "openai>=1.0" ipykernel
```

As células de código abaixo (**Instalação** e **Variáveis Ollama**) repetem os comandos para execução direta no notebook.

### Configuração do LLM

#### Variáveis de ambiente (referência)

| Variável | Obrigatória? | Função |
|----------|----------------|--------|
| `OLLAMA_LOCAL` | Para modo **100% local** com Ollama | Defina `1` (ou `true`) para o notebook usar o servidor Ollama em `http://localhost:11434/v1` sem precisar de chave da OpenAI. |
| `OPENAI_BASE_URL` | Opcional | URL da API no formato OpenAI. Ollama usa `http://localhost:11434/v1` (o código completa com `/v1` se faltar). |
| `OPENAI_API_KEY` | Só na nuvem (OpenAI, etc.) | Chave secreta. Com Ollama local pode omitir (o cliente usa o placeholder `ollama`). |
| `LLM_MODEL` | Opcional | Nome do modelo: ex. `llama3.2`, `mistral`, `gpt-4o-mini`. Se não definir, o padrão com `OLLAMA_LOCAL=1` é `llama3.2`. |
| `SIM_SEED` | Opcional | Inteiro para **sortear** as células iniciais dos pacotes (mesma semente = mesmo cenário). Se **não** for definido, cada execução usa uma **semente aleatória** (o valor aparece na primeira linha da simulação). |

#### Passo a passo: LLM **local** só com Ollama (Windows)

1. Confirme que o Ollama está instalado e **rodando** (ícone na bandeja ou execute `ollama serve`).
2. Baixe um modelo (exemplo leve e bom para JSON):
   ```powershell
   ollama pull llama3.2
   ```
3. Teste no terminal: `ollama run llama3.2` e saia com `/bye`.
4. No **mesmo terminal** onde você vai abrir o Jupyter/VS Code, defina o ambiente **antes** de iniciar o Python:
   ```powershell
   $env:OLLAMA_LOCAL = "1"
   $env:LLM_MODEL = "llama3.2"
   ```
   (Opcional explícito: `$env:OPENAI_BASE_URL = "http://localhost:11434/v1"` — se `OLLAMA_LOCAL=1`, a URL padrão já é essa.)
5. Abra o notebook e execute as células. O coordenador chamará o modelo no seu PC; **nenhum dado de API OpenAI é necessário**.

**Nota:** Este notebook inclui uma **célula Python** que define `OLLAMA_LOCAL`, `LLM_MODEL`, `OPENAI_BASE_URL` e `OPENAI_API_KEY` — ela **substitui** os passos 4–5 acima quando executada antes da simulação.

**Alternativa sem `OLLAMA_LOCAL`:** defina manualmente `$env:OPENAI_API_KEY = "ollama"` e `$env:OPENAI_BASE_URL = "http://localhost:11434/v1"` (a API do Ollama é compatível com o cliente OpenAI e não valida a “chave”).

Se **nenhuma** dessas configurações existir, a simulação usa **modo demonstração** (heurística fixa, sem LLM).


---

## Código: ambiente, agentes e simulação


### Saída da simulação: console e arquivos `.txt`

#### Arquivos de log (`resultado_execucao_*.txt`)

Ao executar o **bloco final** da simulação (lote de 4 runs), o programa grava um arquivo de texto na **mesma pasta** deste notebook:

- **Nome:** `resultado_execucao_YYYYMMDD_HHMMSS.txt` (o sufixo numérico é data e hora da execução, para não sobrescrever execuções anteriores).
- **Conteúdo:** log **completo** das quatro missões — cabeçalho do lote, separadores **Run 1/4 … Run 4/4**, passo a passo (prompt do coordenador na primeira run, decisões de alto nível, movimentos `[n]`, coletas/entregas, `--- Fim ---` por run) e, ao final, o bloco **RESUMO COMPILADO — 4 runs** igual ao do console.

#### Saída no console

Para manter o notebook legível, **no console aparecem somente:**

1. O bloco **RESUMO COMPILADO — 4 runs** (uma linha por run com `SIM_SEED`, passos, sucesso da missão, pacotes iniciais, média/mín/máx de passos e indicador de falha do planejador).
2. Uma mensagem com o **caminho absoluto** do arquivo `.txt` gerado naquela execução.

O **detalhe completo** de todas as runs (incluindo Runs 2, 3 e 4 com o mesmo nível de detalhe da Run 1) está **apenas no `.txt`**. Abra esse arquivo no editor de texto ou no explorador de arquivos para revisar ou anexar ao relatório em PDF, se desejado.


In [2]:
import subprocess
import sys

# Instala na MESMA venv do kernel (evita `pip install jupyter` completo no Windows — caminhos longos)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"],
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "openai>=1.0", "ipykernel"],
)
print("OK: openai e ipykernel instalados neste interpretador:", sys.executable)

OK: openai e ipykernel instalados neste interpretador: c:\Users\luis\anaconda3\envs\nlp_gpu\python.exe


In [ ]:
import os

# --- Ollama 100% local (ajuste o modelo ao que você instalou: ollama pull <nome>) ---
os.environ["OLLAMA_LOCAL"] = "1"
os.environ["LLM_MODEL"] = "llama3.2"
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"
os.environ["OPENAI_API_KEY"] = "ollama"

print("Variáveis para o coordenador LLM (Ollama local):")
print("  OLLAMA_LOCAL =", os.environ.get("OLLAMA_LOCAL"))
print("  LLM_MODEL    =", os.environ.get("LLM_MODEL"))
print("  OPENAI_BASE_URL =", os.environ.get("OPENAI_BASE_URL"))

Variáveis para o coordenador LLM (Ollama local):
  OLLAMA_LOCAL = 1
  LLM_MODEL    = llama3.2
  OPENAI_BASE_URL = http://localhost:11434/v1


In [ ]:
# Execute antes: (1) célula que instala pip, (2) célula "Variáveis Ollama".
# Reinicie o kernel se mudar de venv ou de modelo.

from __future__ import annotations

import json
import os
import random
import re
import sys
from datetime import datetime
from pathlib import Path
from collections import deque
from dataclasses import dataclass, field
from typing import Any, Deque, Dict, List, Optional, Set, Tuple
from openai import OpenAI


GRID_SIZE = 6
DELIVERY: Tuple[int, int] = (5, 5)
OBSTACLES: Set[Tuple[int, int]] = {(2, 2), (2, 3), (4, 4)}
ROBOT_START: Tuple[int, int] = (0, 0)

Move = Tuple[int, int]  # (d_row, d_col)


def random_package_positions(
    seed: int,
    *,
    size: int = GRID_SIZE,
    obstacles: Optional[Set[Tuple[int, int]]] = None,
    delivery: Tuple[int, int] = DELIVERY,
    robot_start: Tuple[int, int] = ROBOT_START,
    count: int = 3,
) -> Set[Tuple[int, int]]:
    """Sorteia células livres para pacotes (reproduzível com a mesma semente)."""
    rng = random.Random(seed)
    obs = set(obstacles) if obstacles is not None else set(OBSTACLES)
    candidates = [
        (r, c)
        for r in range(size)
        for c in range(size)
        if (r, c) not in obs and (r, c) != delivery and (r, c) != robot_start
    ]
    if len(candidates) < count:
        raise ValueError(f"Células livres insuficientes: precisa {count}, há {len(candidates)}")
    return set(rng.sample(candidates, count))


@dataclass
class Environment:
    """Ambiente de grade: obstáculos fixos, pacotes, zona de entrega e robô."""

    size: int = GRID_SIZE
    robot: Tuple[int, int] = ROBOT_START
    carrying: bool = False
    packages: Set[Tuple[int, int]] = field(default_factory=set)
    obstacles: Set[Tuple[int, int]] = field(default_factory=lambda: set(OBSTACLES))
    delivery: Tuple[int, int] = DELIVERY
    step_count: int = 0

    def in_bounds(self, r: int, c: int) -> bool:
        return 0 <= r < self.size and 0 <= c < self.size

    def is_walkable(self, r: int, c: int) -> bool:
        return self.in_bounds(r, c) and (r, c) not in self.obstacles

    def mission_complete(self) -> bool:
        return not self.packages and not self.carrying

    def perceive_text(self) -> str:
        """Descrição textual do estado para o coordenador LLM (percepção)."""
        pk = sorted(self.packages)
        obs = sorted(self.obstacles)
        lines = [
            f"Grade {self.size}x{self.size}, passo de simulação {self.step_count}.",
            f"Robô na célula (linha={self.robot[0]}, coluna={self.robot[1]}).",
            f"Carregando pacote: {self.carrying}.",
            f"Pacotes ainda no armazém (linha,coluna): {pk}.",
            f"Zona de entrega (linha,coluna): {self.delivery}.",
            f"Obstáculos imóveis: {obs}.",
            "Regras: um movimento por passo em {cima, baixo, esquerda, direita}; não atravessar obstáculo; "
            "só pode carregar um pacote por vez; entrega só na zona de entrega.",
        ]
        return "\n".join(lines)

    def structured_state(self) -> Dict[str, Any]:
        return {
            "robot": list(self.robot),
            "carrying": self.carrying,
            "packages": [list(p) for p in sorted(self.packages)],
            "delivery": list(self.delivery),
            "obstacles": [list(o) for o in sorted(self.obstacles)],
            "size": self.size,
        }

    def move_delta(self, drow: int, dcol: int) -> bool:
        r, c = self.robot[0] + drow, self.robot[1] + dcol
        if not self.is_walkable(r, c):
            return False
        self.robot = (r, c)
        self.step_count += 1
        return True

    def pickup_if_applicable(self) -> bool:
        if self.carrying:
            return False
        if self.robot in self.packages:
            self.packages.remove(self.robot)
            self.carrying = True
            return True
        return False

    def deliver_if_applicable(self) -> bool:
        if not self.carrying or self.robot != self.delivery:
            return False
        self.carrying = False
        return True


def bfs_path(
    start: Tuple[int, int],
    goal: Tuple[int, int],
    obstacles: Set[Tuple[int, int]],
    size: int,
) -> Optional[List[Move]]:
    if start == goal:
        return []
    q: Deque[Tuple[Tuple[int, int], List[Move]]] = deque()
    q.append((start, []))
    visited: Set[Tuple[int, int]] = {start}
    while q:
        (r, c), path = q.popleft()
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc = r + dr, c + dc
            if not (0 <= nr < size and 0 <= nc < size):
                continue
            if (nr, nc) in obstacles:
                continue
            if (nr, nc) in visited:
                continue
            visited.add((nr, nc))
            npath = path + [(dr, dc)]
            if (nr, nc) == goal:
                return npath
            q.append(((nr, nc), npath))
    return None


@dataclass
class GoalBasedTransporter:
    """Agente baseado em objetivos: planeja com BFS e executa até o subobjetivo."""

    pending_moves: List[Move] = field(default_factory=list)
    current_purpose: Optional[str] = None  # "pickup" | "deliver"

    def clear_plan(self) -> None:
        self.pending_moves = []
        self.current_purpose = None

    def set_high_level(
        self, env: Environment, goal: Tuple[int, int], purpose: str
    ) -> bool:
        path = bfs_path(env.robot, goal, env.obstacles, env.size)
        if path is None:
            return False
        self.pending_moves = path
        self.current_purpose = purpose
        return True

    def step(self, env: Environment) -> Tuple[str, bool]:
        """Executa um passo de movimento. Retorna (mensagem, moveu?)."""
        if not self.pending_moves:
            return ("sem plano pendente", False)
        dr, dc = self.pending_moves.pop(0)
        ok = env.move_delta(dr, dc)
        if not ok:
            return ("movimento bloqueado (obstáculo/fora)", False)
        name = {( -1, 0): "cima", (1, 0): "baixo", (0, -1): "esquerda", (0, 1): "direita"}.get(
            (dr, dc), "?"
        )
        return (f"moveu {name} -> {env.robot}", True)

    def at_goal(self, goal: Tuple[int, int], env: Environment) -> bool:
        return env.robot == goal and not self.pending_moves

    def complete_objective_at(self, env: Environment, goal: Tuple[int, int]) -> str:
        if env.robot != goal or self.pending_moves:
            return ""
        if self.current_purpose == "pickup":
            if env.pickup_if_applicable():
                msg = f"coletou pacote em {goal}"
            else:
                msg = "objetivo de coleta: sem pacote na célula"
        elif self.current_purpose == "deliver":
            if env.deliver_if_applicable():
                msg = "entregou na zona de entrega"
            else:
                msg = "objetivo entrega: não aplicável"
        else:
            msg = ""
        self.clear_plan()
        return msg


def _extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            return None
    return None


COORDINATOR_SYSTEM = """Você é o agente coordenador de um robô em um armazém em grade.
Responda APENAS com um único objeto JSON válido, sem markdown nem texto extra.
Esquema:
{"purpose": "pickup" | "deliver", "goal": [linha, coluna] }
- Se purpose for "pickup", goal deve ser a célula [linha,coluna] de um pacote listado na percepção.
- Se purpose for "deliver", goal deve ser exatamente a célula da zona de entrega dada na percepção.
- Se já estiver carregando pacote, prefira "deliver" com a célula de entrega.
- Se não houver pacotes e não estiver carregando, use {"purpose":"deliver","goal":<entrega>} apenas se ainda precisar encerrar; caso contrário indique entrega se for o fechamento lógico.
Escolha o pacote mais eficiente (menor deslocamento estimado) quando houver vários.
"""


def _use_ollama_local() -> bool:
    """True se OLLAMA_LOCAL=1 ou se OPENAI_BASE_URL aponta para o servidor Ollama (porta 11434)."""
    if os.environ.get("OLLAMA_LOCAL", "").strip().lower() in ("1", "true", "yes"):
        return True
    base = os.environ.get("OPENAI_BASE_URL") or ""
    return "11434" in base


class LLMCoordinator:
    """Agente cujo processo de decisão usa um LLM (prompt engineering)."""

    def __init__(self) -> None:
        default_model = "llama3.2" if _use_ollama_local() else "gpt-4o-mini"
        self.model = os.environ.get("LLM_MODEL", default_model)
        self._client = None

    def _get_client(self):
        if self._client is not None:
            return self._client
        key = os.environ.get("OPENAI_API_KEY")
        base = (os.environ.get("OPENAI_BASE_URL") or "").strip().rstrip("/")
        if _use_ollama_local():
            key = key or "ollama"
            if not base:
                base = "http://localhost:11434/v1"
            elif not base.endswith("/v1"):
                base = base + "/v1"
        elif not key:
            return None
        kwargs: Dict[str, Any] = {"api_key": key}
        if base:
            kwargs["base_url"] = base
        self._client = OpenAI(**kwargs)
        return self._client

    def decide(self, env: Environment) -> Dict[str, Any]:
        perception = env.perceive_text()
        client = self._get_client()
        if client is None:
            return self._coerce_consistent_command(self._demo_policy(env), env)
        user_msg = f"Percepção atual:\n{perception}\n\nResponda só com o JSON."
        resp = client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": COORDINATOR_SYSTEM},
                {"role": "user", "content": user_msg},
            ],
            temperature=0.2,
        )
        raw = (resp.choices[0].message.content or "").strip()
        parsed = _extract_json_object(raw)
        if parsed is None:
            return self._coerce_consistent_command(self._demo_policy(env), env)
        return self._normalize_command(parsed, env)

    def _normalize_command(self, data: Dict[str, Any], env: Environment) -> Dict[str, Any]:
        purpose = str(data.get("purpose", "")).lower()
        g = data.get("goal")
        if purpose not in ("pickup", "deliver") or not isinstance(g, list) or len(g) != 2:
            return self._coerce_consistent_command(self._demo_policy(env), env)
        r, c = int(g[0]), int(g[1])
        goal = (r, c)
        if purpose == "deliver":
            if goal != env.delivery:
                goal = env.delivery
        elif purpose == "pickup":
            if goal not in env.packages:
                return self._coerce_consistent_command(self._demo_policy(env), env)
        cmd = {"purpose": purpose, "goal": [goal[0], goal[1]]}
        return self._coerce_consistent_command(cmd, env)

    def _coerce_consistent_command(
        self, cmd: Dict[str, Any], env: Environment
    ) -> Dict[str, Any]:
        """Evita loop: entregar sem carga ou buscar pacote já carregando."""
        purpose = str(cmd.get("purpose", "")).lower()
        g = cmd.get("goal")
        if purpose not in ("pickup", "deliver") or not isinstance(g, list) or len(g) != 2:
            return self._demo_policy(env)
        gr, gc = int(g[0]), int(g[1])
        if env.carrying:
            return {
                "purpose": "deliver",
                "goal": [env.delivery[0], env.delivery[1]],
            }
        if purpose == "deliver" and env.packages:
            rx, cy = env.robot
            best = min(
                env.packages,
                key=lambda p: abs(p[0] - rx) + abs(p[1] - cy),
            )
            return {"purpose": "pickup", "goal": [best[0], best[1]]}
        return {"purpose": purpose, "goal": [gr, gc]}

    def _demo_policy(self, env: Environment) -> Dict[str, Any]:
        """Política determinística quando não há API — apenas para teste local."""
        if env.carrying:
            return {"purpose": "deliver", "goal": [env.delivery[0], env.delivery[1]]}
        if env.packages:
            rx, cy = env.robot
            best = min(
                env.packages,
                key=lambda p: abs(p[0] - rx) + abs(p[1] - cy),
            )
            return {"purpose": "pickup", "goal": [best[0], best[1]]}
        return {"purpose": "deliver", "goal": [env.delivery[0], env.delivery[1]]}


def run_simulation(
    max_steps: int = 300,
    verbose: bool = True,
    sim_seed: Optional[int] = None,
    num_packages: int = 3,
    return_stats: bool = False,
    show_prompt_banner: bool = True,
) -> Optional[Dict[str, Any]]:
    if sim_seed is None:
        raw = os.environ.get("SIM_SEED")
        if raw is not None and str(raw).strip() != "":
            sim_seed = int(raw)
        else:
            sim_seed = random.randrange(0, 2**31)
    initial_packages = random_package_positions(
        sim_seed, count=num_packages
    )
    env = Environment(packages=set(initial_packages))
    transporter = GoalBasedTransporter()
    coordinator = LLMCoordinator()
    current_goal: Optional[Tuple[int, int]] = None
    planner_failed = False

    if verbose:
        print("=== Simulação: centro de distribuição em grade ===\n")
        print(
            f"Cenário: SIM_SEED={sim_seed} | pacotes iniciais (lin,col): {sorted(initial_packages)}"
        )
        print(
            f"(Para repetir este cenário: run_simulation(sim_seed={sim_seed}) ou os.environ[\"SIM_SEED\"]=\"{sim_seed}\".)\n"
        )
        if show_prompt_banner:
            print(
                "Prompt de sistema do coordenador LLM (trecho):\n",
                COORDINATOR_SYSTEM[:400],
                "...\n",
            )

    step = 0
    while step < max_steps and not env.mission_complete():
        if not transporter.pending_moves:
            cmd = coordinator.decide(env)
            purpose = cmd["purpose"]
            gr, gc = int(cmd["goal"][0]), int(cmd["goal"][1])
            goal = (gr, gc)
            if verbose:
                print(f"--- Decisão de alto nível (passo {step}) -> {cmd}")
            ok = transporter.set_high_level(env, goal, purpose)
            if not ok:
                planner_failed = True
                if verbose:
                    print("Não foi possível planejar caminho até", goal, "— abortando.")
                break
            current_goal = goal
            if not transporter.pending_moves:
                extra = transporter.complete_objective_at(env, goal)
                if verbose and extra:
                    print(f"  => {extra}")
                current_goal = None
                continue

        msg, moved = transporter.step(env)
        step += 1
        if verbose:
            print(f"  [{env.step_count}] {msg}")

        if current_goal is not None and transporter.at_goal(current_goal, env):
            extra = transporter.complete_objective_at(env, current_goal)
            if verbose and extra:
                print(f"  => {extra}")
            current_goal = None

    if verbose:
        print("\n--- Fim ---")
        print("Missão completa:", env.mission_complete())
        print("Passos de simulação:", env.step_count)
        print("Estado final:", env.structured_state())

    stats: Dict[str, Any] = {
        "sim_seed": sim_seed,
        "mission_complete": env.mission_complete(),
        "steps": env.step_count,
        "initial_packages": sorted(initial_packages),
        "planner_failed": planner_failed,
    }
    if return_stats:
        return stats
    return None


# --- Uma execução isolada (opcional): descomente a linha abaixo ---
# run_simulation()


def _emit_resumo_compilado(batch: List[Dict[str, Any]]) -> None:
    ok_n = sum(1 for r in batch if r["mission_complete"])
    steps_all = [r["steps"] for r in batch]
    print("\n" + "=" * 72)
    print("RESUMO COMPILADO — 4 runs")
    print("=" * 72)
    for i, r in enumerate(batch, start=1):
        print(
            f"  Run {i}: SIM_SEED={r['sim_seed']} | passos={r['steps']} | "
            f"missão_ok={r['mission_complete']} | planner_falhou={r['planner_failed']} | "
            f"pacotes_iniciais={r['initial_packages']}"
        )
    print(f"\n  Missões concluídas: {ok_n}/4")
    print(
        f"  Passos — média: {sum(steps_all)/len(steps_all):.2f} | "
        f"mín: {min(steps_all)} | máx: {max(steps_all)}"
    )
    print(f"  Algum planner falhou: {any(r['planner_failed'] for r in batch)}")
    print("=" * 72)


# --- Lote: grava as 4 runs completas em .txt; no console só Run 1 detalhada + resumo ---
batch_seeds = [random.randrange(0, 2**31) for _ in range(4)]
exec_id = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = Path(f"resultado_execucao_{exec_id}.txt")
batch: List[Dict[str, Any]] = []
_stdout_backup = sys.stdout
with open(out_path, "w", encoding="utf-8") as _logf:
    sys.stdout = _logf
    try:
        print("\n" + "=" * 72)
        print("Lote de 4 execuções — passo a passo de cada missão; tabela resumo ao final.")
        print("=" * 72)
        for run_idx in range(1, 5):
            batch_seed = batch_seeds[run_idx - 1]
            print("\n" + "#" * 72)
            print(f"### Run {run_idx}/4  |  semente fixada para reprodução: {batch_seed}")
            print("#" * 72 + "\n")
            st = run_simulation(
                verbose=True,
                sim_seed=batch_seed,
                return_stats=True,
                show_prompt_banner=(run_idx == 1),
            )
            assert st is not None
            batch.append(st)
        _emit_resumo_compilado(batch)
    finally:
        sys.stdout = _stdout_backup

_emit_resumo_compilado(batch)
print(
    f"\n>>> Log completo das 4 missões (passo a passo de todas as runs):\n"
    f"    {out_path.resolve()}\n"
    "    Abra esse .txt para ver Run 2, 3 e 4 com o mesmo nível de detalhe da Run 1.\n"
)


RESUMO COMPILADO — 4 runs
  Run 1: SIM_SEED=1075043658 | passos=20 | missão_ok=True | planner_falhou=False | pacotes_iniciais=[(2, 5), (3, 5), (4, 1)]
  Run 2: SIM_SEED=1883813619 | passos=22 | missão_ok=True | planner_falhou=False | pacotes_iniciais=[(0, 5), (2, 4), (3, 5)]
  Run 3: SIM_SEED=398558911 | passos=32 | missão_ok=True | planner_falhou=False | pacotes_iniciais=[(1, 1), (1, 4), (3, 1)]
  Run 4: SIM_SEED=1946551642 | passos=32 | missão_ok=True | planner_falhou=False | pacotes_iniciais=[(1, 2), (3, 1), (5, 0)]

  Missões concluídas: 4/4
  Passos — média: 26.50 | mín: 20 | máx: 32
  Algum planner falhou: False

>>> Log completo das 4 missões (passo a passo de todas as runs):
    C:\Users\luis\code\python\pos-grad\agentes-inteligentes\trabalho_final\resultado_execucao_20260401_155440.txt
    Abra esse .txt para ver Run 2, 3 e 4 com o mesmo nível de detalhe da Run 1.

